In [21]:
import pandas as pd

In [22]:
# importation du dataset pour analyse et transformation
df=pd.read_csv('c:/Users/NourdiNE/marketing_dataset.csv')
df

,client_id,age,ville,canal,campagne,montant_achat,conversion,date
0,C1000,56,Biancaberg,SMS,Black Friday,384.48,1,2026-01-03
1,C1001,22,Port Krystalside,SMS,Promo Janvier,69.22,1,2025-09-09
2,C1002,52,Turnermouth,Email,Black Friday,23.71,1,2025-10-21
3,C1003,32,Bryanton,SMS,Black Friday,359.80,1,2026-01-10
4,C1004,59,North Sandra,SMS,Noël,18.76,1,2025-06-20
...,...,...,...,...,...,...,...,...
295,C1295,47,Lake Ashley,SMS,Black Friday,340.46,1,2025-05-25
296,C1296,32,Lake Kenneth,Email,Soldes Été,32.87,1,2025-12-17
297,C1297,29,Allenshire,Facebook,Black Friday,173.32,1,2025-11-24
298,C1298,60,Faulknerberg,Facebook,Noël,162.35,1,2026-01-14


In [23]:
# 1.conversion et enrichissement temporel
df['date']=pd.to_datetime(df['date'])

In [24]:
df['mois']=df['date'].dt.to_period('M').astype(str)

In [25]:
df['annee']=df['date'].dt.year

In [26]:
df['trimestre']=df['date'].dt.quarter

In [27]:
df['jour_semaine']=df['date'].dt.day_name()

In [28]:
# 2.Segmentationn client (VIP,Standard)
CA_total=df.groupby('client_id')['montant_achat'].sum().reset_index()

In [29]:
seuil_vip=CA_total['montant_achat'].quantile(0.80)

In [30]:
CA_total['segment']=CA_total['montant_achat'].apply(lambda x:'VIP' if x >= seuil_vip else'Standard')

In [31]:
# 3.Fusion des infos enrichies
df_final=df.merge(CA_total[['client_id','segment']], on='client_id', how='left')

In [32]:
df_final['tranche_age']=pd.cut(df_final['age'], [0,25,40,60,100], labels=['18-25','26-40','41-60','60+'])

In [33]:
df_final

,client_id,age,ville,canal,campagne,montant_achat,conversion,date,mois,annee,trimestre,jour_semaine,segment,tranche_age
0,C1000,56,Biancaberg,SMS,Black Friday,384.48,1,2026-01-03,2026-01,2026,1,Saturday,Standard,41-60
1,C1001,22,Port Krystalside,SMS,Promo Janvier,69.22,1,2025-09-09,2025-09,2025,3,Tuesday,Standard,18-25
2,C1002,52,Turnermouth,Email,Black Friday,23.71,1,2025-10-21,2025-10,2025,4,Tuesday,Standard,41-60
3,C1003,32,Bryanton,SMS,Black Friday,359.80,1,2026-01-10,2026-01,2026,1,Saturday,Standard,26-40
4,C1004,59,North Sandra,SMS,Noël,18.76,1,2025-06-20,2025-06,2025,2,Friday,Standard,41-60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,C1295,47,Lake Ashley,SMS,Black Friday,340.46,1,2025-05-25,2025-05,2025,2,Sunday,Standard,41-60
296,C1296,32,Lake Kenneth,Email,Soldes Été,32.87,1,2025-12-17,2025-12,2025,4,Wednesday,Standard,26-40
297,C1297,29,Allenshire,Facebook,Black Friday,173.32,1,2025-11-24,2025-11,2025,4,Monday,Standard,26-40
298,C1298,60,Faulknerberg,Facebook,Noël,162.35,1,2026-01-14,2026-01,2026,1,Wednesday,Standard,41-60


In [34]:
df_final.to_csv('data_powerbi.csv',index=False, encoding='utf-8')